<a href="https://colab.research.google.com/github/OLDHOUSE-MECHANIC/ColabNotebookTools-Optimized/blob/main/LTXV_Anime_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎌 LTXV 0.9.5 — Anime Scene Generator
**Run anime video generation on Google Colab (Free T4 GPU)**

### Before you start:
1. Go to **Runtime → Change runtime type → GPU (T4)**
2. Run cells **one by one**, top to bottom
3. Each 5-second clip takes ~60–90 seconds on T4

---

## Cell 1 — Check your GPU

In [6]:
# Check what GPU Colab gave you
!nvidia-smi
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Sat Jun 13 07:21:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P0             26W /   70W |   12387MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cell 2 — Install dependencies
*(Takes about 2–3 minutes. Run once per session.)*

In [7]:
# Install the required libraries
!pip install -q diffusers transformers accelerate sentencepiece
!pip install -q imageio imageio-ffmpeg

print("✅ All dependencies installed!")

✅ All dependencies installed!


## Cell 3 — Load the LTXV 0.9.5 model
*(Downloads ~6 GB the first time — takes 3–5 minutes. Cached on repeat runs.)*

In [2]:
import torch
from diffusers import LTXPipeline
from diffusers.utils import export_to_video
import gc

print("Loading LTXV 0.9.5 model... (this downloads ~6GB first time)")

pipe = LTXPipeline.from_pretrained(
    "Lightricks/LTX-Video-0.9.5",
    torch_dtype=torch.bfloat16
)

# Memory optimizations for T4 (15GB VRAM)
pipe.enable_model_cpu_offload()   # Offloads parts to CPU when not in use
pipe.vae.enable_slicing()         # Saves VRAM during decoding
pipe.vae.enable_tiling()          # Saves VRAM for larger frames

print("✅ Model loaded and ready!")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading LTXV 0.9.5 model... (this downloads ~6GB first time)


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

✅ Model loaded and ready!


## Cell 4 — ✏️ Write your anime prompt & generate!

**Tips for great anime prompts:**
- Be very descriptive (LTX loves detail)
- Include: subject, action, setting, lighting, style, camera angle
- Add `anime style, cel shaded, 2D animation` to push it toward anime
- Always include a negative prompt

**Frame count guide (must follow N×8 + 1 rule):**
| Frames | Duration at 24fps |
|--------|-------------------|
| 49     | ~2 seconds        |
| 97     | ~4 seconds        |
| 121    | ~5 seconds ✅ recommended |
| 161    | ~6.7 seconds      |
| 193    | ~8 seconds (may OOM on T4) |

In [ ]:
import time
from IPython.display import display, Image as DisplayImage, Video
from google.colab import files
from PIL import Image
import io # To handle byte streams from upload

# ============================================================
#  ✏️  EDIT THESE — your anime scene settings
# ============================================================

PROMPT = """
An anime samurai warrior standing on a rooftop at sunset,
wind blowing through his dark hair and flowing black cloak,
cherry blossom petals swirling around him,
city skyline glowing orange and purple behind him,
dramatic low-angle camera shot looking up at the hero,
anime style, cel shaded, 2D animation, Studio Ghibli inspired,
cinematic lighting, sharp detailed linework
""" # @param {type:"string"}

NEGATIVE_PROMPT = """
worst quality, blurry, jittery, distorted, inconsistent motion,
realistic photo, 3D render, low quality, watermark, text
""" # @param {type:"string"}

NUM_FRAMES = 50    # @param {type:"integer"} Recommended for T4: 121 (~5s). Try 97 or 49 for OOM.
WIDTH  = 704        # @param {type:"integer"} Keep these as-is for best quality on T4.
HEIGHT = 480        # @param {type:"integer"} Portrait? Swap to WIDTH=480, HEIGHT=704.
SEED   = 42         # @param {type:"integer"} Change this for different results with same prompt.
STEPS  = 30         # @param {type:"integer"} 20–30 is good. Higher = slower but slightly better.

# --- New User Controls for Initial Frame ---
INIT_FRAME_SOURCE = 'Text-to-Video'         # @param ["Text-to-Video",
                                            #         "Upload Image as First Frame",
                                            #         "Use Last Frame of Previous Video"]

# ============================================================

initial_image = None
pipeline_to_use = pipe # Default to text-to-video pipeline

if INIT_FRAME_SOURCE == 'Upload Image as First Frame':
    print("Please upload your initial image (e.g., PNG, JPG):")
    uploaded = files.upload()
    if uploaded:
        uploaded_filename = list(uploaded.keys())[0]
        initial_image = Image.open(io.BytesIO(uploaded[uploaded_filename]))
        print(f"✅ Initial image '{uploaded_filename}' uploaded and loaded.")
        display(DisplayImage(filename=uploaded_filename, width=200))

        if 'pipe_i2v' in locals() and pipe_i2v is not None:
            pipeline_to_use = pipe_i2v
        else:
            print("❌ 'LTXImageToVideoPipeline' (pipe_i2v) is not loaded. Please run Cell 7 first, or move its loading to Cell 3.")
            print("Falling back to 'Text-to-Video' mode, ignoring uploaded image.")
            INIT_FRAME_SOURCE = 'Text-to-Video'
            initial_image = None
    else:
        print("❌ No image uploaded. Falling back to 'Text-to-Video' mode.")
        INIT_FRAME_SOURCE = 'Text-to-Video'

elif INIT_FRAME_SOURCE == 'Use Last Frame of Previous Video':
    if 'video_frames' in locals() and video_frames is not None:
        initial_image = Image.fromarray(video_frames[-1])
        print("✅ Using the last frame of the previous video as the initial frame.")
        initial_image.save("/content/temp_last_frame_as_init.png") # Save for display
        display(DisplayImage(filename="/content/temp_last_frame_as_init.png", width=200))

        if 'pipe_i2v' in locals() and pipe_i2v is not None:
            pipeline_to_use = pipe_i2v
        else:
            print("❌ 'LTXImageToVideoPipeline' (pipe_i2v) is not loaded. Please run Cell 7 first, or move its loading to Cell 3.")
            print("Falling back to 'Text-to-Video' mode, ignoring previous last frame.")
            INIT_FRAME_SOURCE = 'Text-to-Video'
            initial_image = None
    else:
        print("⚠️ No previous 'video_frames' found (e.g., this is the first run). Falling back to 'Text-to-Video' mode.")
        INIT_FRAME_SOURCE = 'Text-to-Video'


start_time = time.time()
print(f"🎬 Generating your anime scene using {INIT_FRAME_SOURCE} method...")

generator = torch.Generator(device="cpu").manual_seed(SEED)

if initial_image is not None:
    # Use the image-to-video pipeline
    generated_output = pipeline_to_use(
        image=initial_image,
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        width=WIDTH,
        height=HEIGHT,
        num_frames=NUM_FRAMES,
        num_inference_steps=STEPS,
        generator=generator,
    )
else:
    # Use the text-to-video pipeline
    generated_output = pipeline_to_use( # This will be 'pipe'
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        width=WIDTH,
        height=HEIGHT,
        num_frames=NUM_FRAMES,
        num_inference_steps=STEPS,
        generator=generator,
    )

video_frames = generated_output.frames[0] # Store for potential future chaining in Cell 7

elapsed = time.time() - start_time
print(f"✅ Done in {elapsed:.0f} seconds!")

# Save the video
output_path = "/content/anime_scene.mp4"
export_to_video(video_frames, output_path, fps=24)
print(f"💾 Saved to: {output_path}")

# Display for immediate preview
display(Video(output_path, embed=True, width=700))

print("\n--- Additional Notes ---")
print("1. **Target Last Frame:** The current diffusion models primarily generate video from a starting point (text or image). Directly conditioning the video to *end* on a specific uploaded last frame is not a standard feature of these pipelines. If you need a specific end frame, you might consider manual editing or exploring more advanced conditional generation techniques.")
print("2. **Retry Mechanism:** If you are not satisfied with the generated video, simply adjust your `PROMPT`, `NEGATIVE_PROMPT`, `SEED`, or other settings and re-run this cell. If using a previous video's last frame as input, ensure the previous generation was satisfactory.")

🎬 Generating your anime scene using Text-to-Video method...


  0%|          | 0/30 [00:00<?, ?it/s]

## Cell 5 — Preview the video in Colab

In [ ]:
from IPython.display import Video, display
display(Video(output_path, embed=True, width=700))

## Cell 6 — Download the video to your PC

In [ ]:
from google.colab import files
files.download(output_path)
print("📥 Download started!")

---
## 🔗 Cell 7 — Chain clips (extend your scene!)
Generate a second clip that **continues from where the last one ended**.
Run Cell 4 first, then run this cell for a seamless continuation.

In [ ]:
# This uses the LAST FRAME of the previous clip as the starting point
from PIL import Image
import numpy as np
from diffusers import LTXImageToVideoPipeline

# Extract last frame from previous generation
last_frame_array = video_frames[-1]  # numpy array from previous generation
last_frame_pil = Image.fromarray(last_frame_array)
last_frame_pil.save("/content/last_frame.png")
print("Last frame extracted ✅")

# ============================================================
#  ✏️  Continuation prompt — describe what happens NEXT
# ============================================================

CONTINUATION_PROMPT = """
The anime samurai leaps off the rooftop,
diving down toward the city streets below,
cloak billowing dramatically in the wind,
cherry blossoms trailing behind him,
anime style, cel shaded, dynamic action shot
"""

# ============================================================

# Load image-to-video pipeline (reuses same model weights, no re-download)
pipe_i2v = LTXImageToVideoPipeline.from_pretrained(
    "Lightricks/LTX-Video-0.9.5",
    torch_dtype=torch.bfloat16
)
pipe_i2v.enable_model_cpu_offload()
pipe_i2v.vae.enable_slicing()
pipe_i2v.vae.enable_tiling()

print("🎬 Generating continuation clip...")
start = time.time()

continuation_frames = pipe_i2v(
    image=last_frame_pil,
    prompt=CONTINUATION_PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    width=WIDTH,
    height=HEIGHT,
    num_frames=NUM_FRAMES,
    num_inference_steps=STEPS,
    generator=torch.Generator(device="cpu").manual_seed(SEED + 1),
).frames[0]

print(f"✅ Done in {time.time()-start:.0f}s")

# Save continuation clip
clip2_path = "/content/anime_scene_part2.mp4"
export_to_video(continuation_frames, clip2_path, fps=24)
print(f"💾 Saved: {clip2_path}")

display(Video(clip2_path, embed=True, width=700))

## Cell 8 — Merge all clips into one video

In [ ]:
# Combine Part 1 + Part 2 into a single video using ffmpeg
!echo "file '/content/anime_scene.mp4'" > /content/filelist.txt
!echo "file '/content/anime_scene_part2.mp4'" >> /content/filelist.txt
!ffmpeg -y -f concat -safe 0 -i /content/filelist.txt -c copy /content/anime_full.mp4

print("✅ Merged! Full scene:")
display(Video("/content/anime_full.mp4", embed=True, width=700))

# Download merged video
files.download("/content/anime_full.mp4")

---
## 💡 Prompt Templates — Copy & Paste

### Action scene
```
anime ninja running across rooftops at night, jumping between buildings,
neon city lights below, rain falling, motion blur on fists,
dynamic side-scrolling camera, anime style, 2D cel shaded,
dramatic action cinematography, sharp linework
```

### Emotional moment
```
close-up of anime girl's face, tears slowly falling,
soft golden light from a window, cherry blossoms visible through glass,
slow camera push-in, anime style, Studio Ghibli inspired,
pastel color palette, gentle melancholic mood
```

### Landscape / establishing shot  
```
wide shot of a futuristic anime city at dusk,
giant holographic billboards, flying vehicles in the sky,
camera slowly panning left revealing the skyline,
cyberpunk anime style, cel shaded, vivid neon colors,
cinematic establishing shot
```

### Battle scene
```
two anime warriors clashing swords mid-air,
energy shockwave exploding outward, dramatic lighting,
slow motion shot of sparks flying between blades,
anime style, dynamic composition, intense battle atmosphere,
bold linework and vivid colors
```

---
## ⚠️ Troubleshooting
| Problem | Fix |
|---------|-----|
| Out of memory (OOM) | Reduce NUM_FRAMES to 97 or 49 |
| Black video output | Frame count must be N×8+1 (49, 97, 121, 161...) |
| Session disconnected | Re-run from Cell 3 (model reloads, ~5 min) |
| Looks too realistic | Add more anime keywords: `anime style, 2D cel shaded, flat shading` |
| Bad motion / jitter | Lower NUM_FRAMES, or add `smooth motion` to prompt |